[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/27_vit_patch_interview.ipynb)

# 🟠 Solution: ViT Patch Embedding（面试版）

## 📋 题目描述

**难度：** Medium

实现 ViT 图像块嵌入（nn.Module）。

图像块嵌入将图像分割为固定大小的块，并将每个块投影为嵌入向量，形成 Vision Transformer 的输入序列。

**签名:** `PatchEmbedding(img_size, patch_size, in_channels, embed_dim)`（nn.Module）

**前向传播:** `forward(x) -> Tensor`
- `x` — 图像张量 (B, C, H, W)

**返回:** 图像块嵌入 (B, num_patches, embed_dim)

**约束:**
- `num_patches = (img_size / patch_size)^2`
- 将 `self.num_patches` 存储为属性
- 使用 `nn.Linear(C * P * P, embed_dim)` 投影

**提示：** `(B,C,H,W)` → reshape 为 `(B, n_h, P, n_w, P, C)` → permute → `(B, num_patches, C*P*P)` → `nn.Linear(C*P*P, embed_dim)`。


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ INTERVIEW

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Linear(in_channels * patch_size * patch_size, embed_dim)

    def forward(self, x):
        B, C, H, W = x.shape
        p = self.patch_size
        n_h, n_w = H // p, W // p

        # ---- 手写切分 + 展平 ----
        # 面试要点：reshape + permute 的顺序
        #
        # 目标：把 (B,C,H,W) 切成 (B, num_patches, C*P*P)
        #
        # 步骤拆解：
        # 1. (B, C, H, W)
        #    → (B, C, n_h, p, n_w, p)   # 把 H/W 拆成"有几个patch"和"patch内大小"
        #    → (B, n_h, n_w, C, p, p)    # permute: patch坐标提到前面
        #    → (B, n_h*n_w, C*p*p)       # reshape: 展平为序列
        #
        # 为什么这样 permute？
        # 我们想要 patch 按"先行后列"的顺序排列
        # (n_h, n_w) 就是先行后列的坐标
        x = x.reshape(B, C, n_h, p, n_w, p)
        x = x.permute(0, 2, 4, 1, 3, 5)
        x = x.reshape(B, n_h * n_w, C * p * p)

        return self.proj(x)

# 面试常见追问：
# Q: 为什么用 Linear 而不是 Conv2d？
# A: 等价！nn.Conv2d(C, embed_dim, kernel_size=patch_size, stride=patch_size)
#    效果完全一样。Conv2d 版本更简洁但面试要求手写切分逻辑
# Q: [CLS] token 和位置编码在哪加？
# A: PatchEmbedding 之后。ViT 在 patch 序列前拼接 [CLS] token
#    然后加上可学习的位置编码
# Q: num_patches 对模型的影响？
# A: 序列越长，自注意力的 O(n²) 开销越大


In [ ]:
pe = PatchEmbedding(img_size=32, patch_size=4, in_channels=3, embed_dim=64)
x = torch.randn(2, 3, 32, 32)
out = pe(x)
print(f'num_patches: {pe.num_patches}, Output: {out.shape}')


In [ ]:
from torch_judge import check
check('vit_patch')


## 📝 核心思路总结

1. **图像切分**：reshape + permute 将 (B,C,H,W) 切成 (B, num_patches, C*P*P)
2. **维度重排**：`(C,n_h,p,n_w,p)` → `(n_h,n_w,C,p,p)` 确保 patch 按行列顺序
3. **线性投影**：展平后的 patch 通过 Linear 映射到 embed_dim
4. **等价 Conv2d**：`nn.Conv2d(C, D, P, stride=P)` 效果相同
